# How orthogonal is the GO:BP training prior to the evaluation gene sets?

CLAMPfull is trained with a GO:BP prior and then evaluated against gene sets from
other collections: CellMarker cell types, the Allen Brain Atlas, and GTEx tissue
signatures. If any of those evaluation sets were near-duplicates of a training term,
recovering them would be circular. This notebook measures that directly.

For every evaluation term we find the single most similar GO:BP term and report the
maximum Jaccard index and maximum overlap coefficient. Jaccard is computed against the
term's **full** size, including genes absent from the GO universe: those genes can
never intersect but do enlarge the union, and keeping them is what makes the measure
honest rather than flattering.

A size- and coverage-matched null accompanies each term. It resamples the same number
of GO-annotated genes at random, so it answers "how similar would *any* set of this
shape be to its best GO:BP match by chance". It is retained in the per-term output;
the supplementary panel plots the observed distribution only.

The prior is shared by the pseudobulk and GTEx models, so this result is not specific
to GTEx; the notebook lives here because the GTEx supplement consumes it.

💡 **Environment:** `clamp-analyses`

## Libraries

In [ ]:
suppressPackageStartupMessages({
  library(here); library(data.table); library(readxl); library(Matrix)
})

## Settings

In [ ]:
OUT_DIR <- here(snakemake@params[["out_dir"]])
dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)

N_NULL <- 100
MIN_GS <- 10; MAX_GS <- 500   # same size window the marker ORA uses
CHUNK  <- 1000
set.seed(42)

## The training prior

This reads the GMT CLAMPfull was actually trained with, so the comparison is against
the training input rather than a re-downloaded copy.

In [ ]:
read_gmt <- function(path) {
  ln <- readLines(path, warn = FALSE)
  sp <- strsplit(ln, "\t", fixed = TRUE)
  sp <- sp[lengths(sp) >= 3]
  # Enrichr GMTs sometimes carry a ",1.0" weight suffix on each gene symbol.
  setNames(lapply(sp, function(x) unique(sub(",.*$", "", x[-c(1, 2)]))),
           vapply(sp, `[`, character(1), 1))
}

go <- read_gmt(snakemake@input[["go_bp_file"]])
go_names <- names(go)
go_univ  <- sort(unique(unlist(go)))
go_idx   <- setNames(seq_along(go_univ), go_univ)
n_go_genes <- length(go_univ)

GO <- sparseMatrix(i = rep(seq_along(go), lengths(go)),
                   j = unname(go_idx[unlist(go)]),
                   x = 1, dims = c(length(go), n_go_genes))
go_sizes <- rowSums(GO)
cat(sprintf("GO:BP prior: %d terms, %d genes, %d memberships, median term size %.0f\n",
            nrow(GO), n_go_genes, length(GO@x), median(go_sizes)))

## The evaluation collections

In [ ]:
cm <- as.data.table(read_excel(snakemake@input[["cell_marker_file"]], sheet = "human"))
cm <- cm[!is.na(Symbol) & !is.na(cell_name) & species == "Human"]
cellmarker <- lapply(split(cm$Symbol, cm$cell_name), unique)
cellmarker <- cellmarker[lengths(cellmarker) >= MIN_GS & lengths(cellmarker) <= MAX_GS]

# Allen: human clusters only. The "down" sets are depletions, not signatures.
allen_all <- read_gmt(snakemake@input[["allen_brain_gmt_file"]])
allen <- allen_all[startsWith(names(allen_all), "Human")]
allen <- allen[lengths(allen) >= MIN_GS & lengths(allen) <= MAX_GS]

# GTEx tissue signatures: a binary genes x tissue indicator matrix.
gm <- readRDS(snakemake@input[["gtex_tissues_pathmat"]])
gtex <- apply(gm, 2, function(col) rownames(gm)[col > 0], simplify = FALSE)
gtex <- gtex[lengths(gtex) >= MIN_GS & lengths(gtex) <= MAX_GS]

collections <- list("GTEx Tissues" = gtex,
                    "CellMarker Human" = cellmarker,
                    "Allen Brain Atlas" = allen)
for (nm in names(collections))
  cat(sprintf("%-20s %5d terms, median size %.0f\n", nm, length(collections[[nm]]),
              median(lengths(collections[[nm]]))))

## Maximum similarity against any GO:BP term

In [ ]:
# sets_idx: gene indices already mapped into the GO universe.
# set_sizes: the FULL set size, including genes outside that universe. Those genes
# can never intersect but do enlarge the union, and keeping them is what stops
# Jaccard from being flattered by poor annotation coverage.
max_similarity <- function(sets_idx, set_sizes) {
  n <- length(sets_idx)
  A <- sparseMatrix(i = rep(seq_len(n), lengths(sets_idx)),
                    j = unlist(sets_idx), x = 1,
                    dims = c(n, n_go_genes))
  best_j <- numeric(n); best_o <- numeric(n); best_i <- integer(n)
  for (s in seq(1L, n, by = CHUNK)) {
    e <- min(s + CHUNK - 1L, n)
    inter <- as.matrix(tcrossprod(A[s:e, , drop = FALSE], GO))
    sz    <- set_sizes[s:e]
    uni   <- outer(sz, go_sizes, "+") - inter
    jac   <- ifelse(uni > 0, inter / uni, 0)
    mins  <- outer(sz, go_sizes, pmin)
    ovl   <- ifelse(mins > 0, inter / mins, 0)
    mi <- max.col(jac, ties.method = "first")
    best_j[s:e] <- jac[cbind(seq_len(nrow(jac)), mi)]
    best_i[s:e] <- mi
    best_o[s:e] <- do.call(pmax, lapply(seq_len(ncol(ovl)), function(k) ovl[, k]))
  }
  list(jaccard = best_j, overlap = best_o, best = best_i)
}

In [ ]:
per_term <- list(); summary <- list()

for (label in names(collections)) {
  ev    <- collections[[label]]
  names_ev <- names(ev)
  sizes <- as.numeric(lengths(ev))
  mapped <- lapply(ev, function(g) unname(go_idx[intersect(g, go_univ)]))
  kcov  <- as.numeric(lengths(mapped))

  obs <- max_similarity(mapped, sizes)

  # Null: same total size, same number of GO-annotated genes, random identity.
  # Only the k in-universe genes can intersect, so only those are resampled.
  null_sets  <- vector("list", length(ev) * N_NULL)
  null_sizes <- numeric(length(ev) * N_NULL)
  owner      <- integer(length(ev) * N_NULL)
  p <- 0L
  for (i in seq_along(ev)) {
    k <- as.integer(kcov[i])
    for (b in seq_len(N_NULL)) {
      p <- p + 1L
      null_sets[[p]]  <- if (k) sample.int(n_go_genes, k) else integer(0)
      null_sizes[p]   <- sizes[i]
      owner[p]        <- i
    }
  }
  nul <- max_similarity(null_sets, null_sizes)
  nj  <- nul$jaccard

  null_med <- tapply(nj, owner, median)
  null_p95 <- tapply(nj, owner, quantile, probs = 0.95)
  emp_p    <- vapply(seq_along(ev), function(i) {
    v <- nj[owner == i]; (sum(v >= obs$jaccard[i]) + 1) / (length(v) + 1)
  }, numeric(1))

  per_term[[label]] <- data.table(
    collection = label, eval_term = names_ev,
    n_genes = as.integer(sizes),
    n_genes_in_go_universe = as.integer(kcov),
    frac_genes_in_go_universe = round(kcov / sizes, 4),
    max_jaccard = round(obs$jaccard, 4),
    max_overlap_coef = round(obs$overlap, 4),
    best_go_term = go_names[obs$best],
    null_median_max_jaccard = round(as.numeric(null_med), 4),
    null_p95_max_jaccard = round(as.numeric(null_p95), 4),
    empirical_p = round(emp_p, 4))

  summary[[label]] <- data.table(
    collection = label, n_terms = length(ev),
    median_set_size = as.integer(median(sizes)),
    median_frac_in_go_universe = round(median(kcov / sizes), 3),
    mean_max_jaccard = round(mean(obs$jaccard), 4),
    median_max_jaccard = round(median(obs$jaccard), 4),
    p90_max_jaccard = round(as.numeric(quantile(obs$jaccard, 0.9)), 4),
    max_max_jaccard = round(max(obs$jaccard), 4),
    median_null_max_jaccard = round(median(as.numeric(null_med)), 4),
    n_jaccard_gt_0.5 = sum(obs$jaccard > 0.5),
    median_max_overlap_coef = round(median(obs$overlap), 4),
    n_overlap_coef_gt_0.8 = sum(obs$overlap > 0.8))
  cat(sprintf("  %-20s done (%d terms)\n", label, length(ev)))
}

per_term <- rbindlist(per_term)
summary  <- rbindlist(summary)

## Results

In [ ]:
print(as.data.frame(summary), row.names = FALSE)
cat(sprintf("\nAcross all %d evaluation terms, %d exceed Jaccard 0.5 and %d exceed overlap coefficient 0.8.\n",
            nrow(per_term), sum(per_term$max_jaccard > 0.5),
            sum(per_term$max_overlap_coef > 0.8)))
cat("\nTop 10 closest matches to any GO:BP term:\n")
print(as.data.frame(per_term[order(-max_jaccard)][1:10,
      .(collection, eval_term, n_genes, max_jaccard, max_overlap_coef, best_go_term)]),
      row.names = FALSE)

## Figure preview

One density per evaluation collection, over each term's maximum Jaccard against any
GO:BP training term. Complete redundancy with a training term would sit at 1.0, so
the whole claim is that these distributions are pressed against the left edge.

The size- and coverage-matched null is kept in the per-term CSV but is not drawn:
plotting it would imply the observed values are close enough to need a null to
interpret, and at a maximum of 0.27 across 1002 terms they are not.

In [ ]:
suppressPackageStartupMessages(library(ggplot2))

ortho <- copy(per_term)
ortho[, collection := factor(collection,
      levels = c("GTEx Tissues", "CellMarker Human", "Allen Brain Atlas"))]
stopifnot(!any(is.na(ortho$collection)))

ortho_means <- ortho[, .(m = mean(max_jaccard), n = .N), by = collection]
ortho_means[, lab := sprintf("mean %.3f (n = %d)", m, n)]

p_ortho <- ggplot(ortho, aes(x = max_jaccard)) +
  geom_density(fill = "#2166AC", colour = "#2166AC", alpha = 0.35, adjust = 1.2) +
  geom_vline(data = ortho_means, aes(xintercept = m),
             linetype = "dashed", linewidth = 0.4, colour = "black") +
  geom_text(data = ortho_means, aes(x = m, y = Inf, label = lab),
            hjust = -0.08, vjust = 1.6, size = 2.8, colour = "black") +
  facet_wrap(~collection, scales = "free_y", ncol = 1) +
  coord_cartesian(xlim = c(0, 0.45)) +
  labs(x = "Max Jaccard against any GO:BP training term", y = "Density",
       title = "Evaluation gene sets vs the GO:BP training prior",
       subtitle = "Complete redundancy with a training term would sit at 1.0") +
  theme_classic(base_size = 10) +
  theme(strip.background = element_blank(),
        strip.text = element_text(face = "bold", size = 9),
        plot.subtitle = element_text(colour = "grey35", size = 8),
        plot.background = element_rect(fill = "white", colour = NA))

options(repr.plot.width = 6, repr.plot.height = 6)
p_ortho

## Save

In [ ]:
fwrite(per_term, snakemake@output[["per_term"]])
fwrite(summary,  snakemake@output[["summary"]])
cat("\nWrote:\n")
for (k in c("per_term", "summary"))
  cat("  ", basename(snakemake@output[[k]]), "\n", sep = "")